# Chapitre 11 - Lab : capture des changements avec le décodage logique de PostgreSQL

Le cours 3 a implémenté une CDC jouet par déclencheurs sur SQLite. Les outils réels
(Debezium et consorts) lisent plutôt le **journal d'écriture anticipée** (WAL) de la base,
décodé en changements logiques. Ce lab utilise directement ce mécanisme dans
PostgreSQL 16 : un **slot de réplication logique** avec le module de sortie
`test_decoding`, interrogé en SQL (`pg_logical_slot_get_changes`). On y retrouve, sur un
vrai système, les notions du cours : ordre de validation, position dans le journal (LSN),
système dérivé qui suit la base, instantané initial cohérent, et retard d'un consommateur.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/streams/compose.yml -p ddia_streams up -d --wait
# ... exécuter le notebook ...
docker compose -f docker/streams/compose.yml -p ddia_streams down -v
```

PostgreSQL écoute sur le port **11232** de l'hôte (utilisateur et mot de passe `ddia`, base
`flux`), avec `wal_level=logical`.

**Plan.**

1. Un slot de réplication logique : le journal des changements en SQL.
2. L'ordre du journal est l'ordre de validation.
3. Ce que contient un changement : `REPLICA IDENTITY`.
4. Un index dérivé qui suit la base, face à la double écriture.
5. L'instantané initial cohérent.
6. Un slot retient le WAL : le retard du consommateur a un coût.

In [1]:
import re
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg

from ddia import streams as st
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.max_colwidth", 200)
DSN = "host=localhost port=11232 user=ddia password=ddia dbname=flux connect_timeout=5"

def sql(requete, params=None):
    with psycopg.connect(DSN, autocommit=True) as c:
        cur = c.execute(requete, params)
        return cur.fetchall() if cur.description else None

print(sql("SELECT version()")[0][0].split(",")[0], "; wal_level =", sql("SHOW wal_level")[0][0])
for (slot,) in sql("SELECT slot_name FROM pg_replication_slots"):
    sql("SELECT pg_drop_replication_slot(%s)", (slot,))            # notebook relançable
sql("DROP TABLE IF EXISTS produits")
sql("CREATE TABLE produits (id INTEGER PRIMARY KEY, nom TEXT, description TEXT, prix NUMERIC)")

PostgreSQL 16.14 (Debian 16.14-1.pgdg13+1) on x86_64-pc-linux-gnu ; wal_level = logical


## 1. Un slot de réplication logique (sections *Change Data Capture* et *Implementing change data capture*)

Un **slot** mémorise la position d'un consommateur dans le WAL, comme l'offset validé
d'un groupe Kafka. `pg_logical_slot_peek_changes` lit les changements sans avancer la
position ; `pg_logical_slot_get_changes` les lit **et** avance la position (on « valide »).
Chaque changement porte son **LSN** (log sequence number), sa position dans le WAL, et
l'identifiant de la transaction (`xid`).

In [2]:
sql("SELECT pg_create_logical_replication_slot('cdc_demo', 'test_decoding')")
sql("INSERT INTO produits VALUES (1, 'Lampe', 'lampe de bureau en laiton', 49)")
sql("INSERT INTO produits VALUES (2, 'Chaise', 'chaise en chêne massif', 120)")
sql("UPDATE produits SET prix = 45 WHERE id = 1")
sql("DELETE FROM produits WHERE id = 2")
apercu = sql("SELECT lsn::text, xid::text, data FROM pg_logical_slot_peek_changes('cdc_demo', NULL, NULL)")
print(f"{len(apercu)} lignes lues sans avancer ; relues : {len(sql('SELECT * FROM pg_logical_slot_peek_changes(%s, NULL, NULL)', ('cdc_demo',)))}")
pd.DataFrame(apercu, columns=["lsn", "xid", "changement (test_decoding)"])

12 lignes lues sans avancer ; relues : 12


,lsn,xid,changement (test_decoding)
0,0/1B964F0,4963,BEGIN 4963
1,0/1B964F0,4963,table public.produits: INSERT: id[integer]:1 nom[text]:'Lampe' description[text]:'lampe de bureau en laiton' prix[numeric]:49
2,0/1B96620,4963,COMMIT 4963
3,0/1B96620,4964,BEGIN 4964
4,0/1B96620,4964,table public.produits: INSERT: id[integer]:2 nom[text]:'Chaise' description[text]:'chaise en chêne massif' prix[numeric]:120
5,0/1B966F0,4964,COMMIT 4964
6,0/1B966F0,4965,BEGIN 4965
7,0/1B966F0,4965,table public.produits: UPDATE: id[integer]:1 nom[text]:'Lampe' description[text]:'lampe de bureau en laiton' prix[numeric]:45
8,0/1B96790,4965,COMMIT 4965
9,0/1B96790,4966,BEGIN 4966


Chaque instruction exécutée en mode autocommit est une transaction : le journal montre
des blocs `BEGIN ... COMMIT`, et dedans un changement par ligne touchée, au format texte
de `test_decoding` (les outils de production utilisent `pgoutput`, binaire, mais
l'information est la même). Écrivons un petit analyseur, puis consommons pour de bon.

In [3]:
MOTIF = re.compile(r"table public\.(\w+): (INSERT|UPDATE|DELETE): (.*)")
CHAMP = re.compile(r"(\w+)\[[^\]]+\]:('(?:[^']|'')*'|\S+)")

def analyser(data):
    """Ligne test_decoding -> dict (table, op, ligne), ou None pour BEGIN/COMMIT."""
    m = MOTIF.match(data)
    if not m:
        return None
    table, op, reste = m.groups()
    champs = {k: (v[1:-1].replace("''", "'") if v.startswith("'") else (None if v == "null" else v))
              for k, v in CHAMP.findall(reste)}
    return {"table": table, "op": op.lower(), "ligne": champs}

def consommer(slot="cdc_demo", limite=None):
    rows = sql("SELECT lsn::text, xid::text, data FROM pg_logical_slot_get_changes(%s, NULL, %s)", (slot, limite))
    return [dict(lsn=l, xid=int(x), brut=d, **c) for l, x, d in rows if (c := analyser(d))]

pd.DataFrame(consommer()).drop(columns="brut")

,lsn,xid,table,op,ligne
0,0/1B964F0,4963,produits,insert,"{'id': '1', 'nom': 'Lampe', 'description': 'lampe de bureau en laiton', 'prix': '49'}"
1,0/1B96620,4964,produits,insert,"{'id': '2', 'nom': 'Chaise', 'description': 'chaise en chêne massif', 'prix': '120'}"
2,0/1B966F0,4965,produits,update,"{'id': '1', 'nom': 'Lampe', 'description': 'lampe de bureau en laiton', 'prix': '45'}"
3,0/1B96790,4966,produits,delete,{'id': '2'}


Après `get_changes`, le slot a avancé : une nouvelle lecture ne renvoie plus rien tant
qu'il n'y a pas de nouvelles écritures.

> **À retenir.** Un slot de réplication logique expose le WAL comme un flux de
> changements ligne à ligne, chacun avec sa position (LSN) ; le slot mémorise la position
> du consommateur.

In [4]:
print("relecture après consommation :", consommer())

relecture après consommation : []
